# **Feature Engineering**

### Objective

Create candidate predictors for the Tanzania Tourism Classification task using trip duration, recorded group size, group composition, and geographic origin. The target is `cost_category`.

### Feature Engineering Approach

Composition is represented by shares of a total. For valid non-negative counts and a positive total, shares range from 0 to 1. A value of 0.75 represents 75%.

Both complementary shares are retained for exploration. For modeling, one share from each pair is sufficient because the other equals one minus that share. Keep the totals to distinguish groups and trips of different sizes.

This notebook constructs and inspects features; it does not establish a model-performance improvement.

## **1. Import Required Libraries & Modules**

NumPy and pandas support numerical transformations and dataframe inspection. Project paths and the shared preprocessing pipeline are imported from `src`. Run the notebook within the project with its configured data files available.

In [4]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import seaborn as sns

from pathlib import Path
import sys 

notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH
from src.preprocessing import Data_Preprocessor


## **2. Load Cleaned Datasets via Preprocessor**

The preprocessing pipeline loads training and competition test data. The training schema includes `cost_category`, while the test schema does not. Inspect the displayed shapes and columns before creating features.

In [5]:
train_df_processed = Data_Preprocessor(expected_columns=21, is_train=True)
train_processed = train_df_processed.process_pipeline(str(TRAIN_DATA_PATH))

test_df_processed = Data_Preprocessor(expected_columns=20, is_train=False)
test_processed = test_df_processed.process_pipeline(str(TEST_DATA_PATH))

print(f"train shape {train_processed.shape}")
print(f"test shape {test_processed.shape}")

train shape (18478, 21)
test shape (6169, 20)


In [6]:
train_processed

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18473,tour_idmp5ciw50,KENYA,45-64,Alone,0.0,1.0,Other,Hunting Tourism,Others,Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
18474,tour_ideq0yncfs,KENYA,45-64,Alone,1.0,0.0,Meetings and Conference,Wildlife Tourism,Others,Independent,...,No,No,No,No,No,No,2,0,No,Lower Cost
18475,tour_idv7pz3vs8,UNITED STATES OF AMERICA,25-44,With Spouse and Children,2.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Independent,...,No,No,No,No,No,No,9,0,Yes,Higher Cost
18476,tour_idy6ydo00w,UNITED STATES OF AMERICA,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,13,4,No,Higher Cost


In [7]:
train_processed.columns

Index(['Tour_ID', 'country', 'age_group', 'travel_with', 'total_female',
       'total_male', 'purpose', 'main_activity', 'info_source',
       'tour_arrangement', 'package_transport_int', 'package_accomodation',
       'package_food', 'package_transport_tz', 'package_sightseeing',
       'package_guided_tour', 'package_insurance', 'night_mainland',
       'night_zanzibar', 'first_trip_tz', 'cost_category'],
      dtype='object')

## **3. Feature Engineering**

Apply identical definitions to `train_processed` and `test_processed`. These transformations use predictor values only and retain the original columns.

Run all cells in order to calculate the features and inspect the resulting dataframes.

### 3.1 Total Nights

**Definition:** `total nights = night_mainland + night_zanzibar`.

This measures combined trip duration and supplies the denominator for destination shares and the group-size-per-night interaction. The column name `total nights` contains a space.

In [8]:
for df in (train_processed, test_processed):
    df.drop(columns=['night_ratio', 'male_to_female_ratio'], errors='ignore', inplace=True)
    df['total nights'] = df['night_mainland'] + df['night_zanzibar']

### 3.2 Total Recorded Group Size

**Definition:** `total_adults = total_female + total_male`.

Confirm from the dataset documentation whether these counts describe adults specifically or all recorded travelers. Keeping the total preserves group-size information that shares alone cannot represent.

In [9]:
for df in (train_processed, test_processed):
    df['total_adults'] = df['total_female'] + df['total_male']

### 3.3 Destination Night Shares

**Definitions:**

- `mainland_nights_share = night_mainland / total nights`.
- `zanzibar_nights_share = night_zanzibar / total nights`.

For example, 6 mainland nights and 2 Zanzibar nights produce shares of 0.75 and 0.25. A mainland-only trip has shares of 1 and 0, while a Zanzibar-only trip has shares of 0 and 1.

When total nights is zero, both shares are undefined and stored as `NaN`. This differs from a valid zero share, which means that destination has no nights within a trip with a positive duration. Missing source values also propagate. Full precision is retained; round only for display if needed.

In [10]:
for df in (train_processed, test_processed):
    nights_denominator = df['total nights'].replace(0, np.nan)
    df['mainland_nights_share'] = (df['night_mainland'] / nights_denominator).round(2)
    df['zanzibar_nights_share'] = (df['night_zanzibar'] / nights_denominator).round(2)

### 3.4 Recorded Group Size per Night

**Definition:** `adults_per_night = total_adults / total nights`.

This is a candidate interaction between group size and trip duration. It is not a composition share and is not limited to the range 0 to 1. Its usefulness must be evaluated separately.

Zero total nights is replaced by `NaN` before division, preventing infinity from division by zero. Full precision is retained.

In [11]:
for df in (train_processed, test_processed):
    nights_denominator = df['total nights'].replace(0, np.nan)
    df['adults_per_night'] = (df['total_adults'] / nights_denominator).round(2)

### 3.5 Male and Female Shares

**Definitions:**

- `male_share = total_male / total_adults`.
- `female_share = total_female / total_adults`.

For 3 males and 1 female, the shares are 0.75 and 0.25. All-male and all-female groups produce (1, 0) and (0, 1), respectively.

When the total group size is zero, both shares are undefined and stored as `NaN`. For valid non-negative counts with a positive total, the two shares sum to 1.

In [12]:
for df in (train_processed, test_processed):
    people_denominator = df['total_adults'].replace(0, np.nan)
    df['male_share'] = (df['total_male'] / people_denominator).round(2)
    df['female_share'] = (df['total_female'] / people_denominator).round(2)

### 3.6 Continent

A fixed lookup groups countries into broader geographic categories. `country_dic` inventories the training labels; its integer keys are inspection indices, not model encodings.

The same mapping is applied to training and test data. Unmapped countries become `NaN`; review coverage on each dataset and define a consistent unknown-category policy before modeling. Geographic assignments should be reviewed separately before production use.

In [27]:
country_dic = dict(enumerate(pd.concat([train_processed['country'], test_processed['country']], axis=0, ignore_index=True).unique()))


In [28]:
country_dic

{0: 'ITALY',
 1: 'UNITED KINGDOM',
 2: 'UNITED STATES OF AMERICA',
 3: 'RWANDA',
 4: 'FRANCE',
 5: 'SWITZERLAND',
 6: 'NIGERIA',
 7: 'AUSTRALIA',
 8: 'SPAIN',
 9: 'DENMARK',
 10: 'BELGIUM',
 11: 'ZAMBIA',
 12: 'NETHERLANDS',
 13: 'CANADA',
 14: 'INDIA',
 15: 'NORWAY',
 16: 'ZIMBABWE',
 17: 'MOROCCO',
 18: 'CHINA',
 19: 'GERMANY',
 20: 'SOUTH AFRICA',
 21: 'JAPAN',
 22: 'UAE',
 23: 'RUSSIA',
 24: 'KENYA',
 25: 'GREECE',
 26: 'POLAND',
 27: 'AUSTRIA',
 28: 'MALAYSIA',
 29: 'CZECH REPUBLIC',
 30: 'KOREA',
 31: 'DRC',
 32: 'CYPRUS',
 33: 'KUWAIT',
 34: 'BRAZIL',
 35: 'CHILE',
 36: 'BULGARIA',
 37: 'ISRAEL',
 38: 'NEW ZEALAND',
 39: 'UGANDA',
 40: 'IRELAND',
 41: 'SWEDEN',
 42: 'QATAR',
 43: 'CONGO',
 44: 'BURUNDI',
 45: 'FINLAND',
 46: 'GHANA',
 47: 'OMAN',
 48: 'LEBANON',
 49: 'LUXEMBOURG',
 50: 'IVORY COAST',
 51: 'NAMIBIA',
 52: 'ROMANIA',
 53: 'THAILAND',
 54: 'SUDAN',
 55: 'SINGAPORE',
 56: 'PORTUGAL',
 57: 'MALAWI',
 58: 'MALTA',
 59: 'EGYPT',
 60: 'PAKISTAN',
 61: 'TANZANIA',
 62: '

In [36]:
continent_mapping = {
    'ITALY': 'EUROPE',
    'UNITED KINGDOM': 'EUROPE',
    'UNITED STATES OF AMERICA': 'NORTH AMERICA',
    'RWANDA': 'AFRICA',
    'FRANCE': 'EUROPE',
    'SWITZERLAND': 'EUROPE',
    'NIGERIA': 'AFRICA',
    'AUSTRALIA': 'OCEANIA',
    'SPAIN': 'EUROPE',
    'DENMARK': 'EUROPE',
    'BELGIUM': 'EUROPE',
    'ZAMBIA': 'AFRICA',
    'NETHERLANDS': 'EUROPE',
    'CANADA': 'NORTH AMERICA',
    'INDIA': 'ASIA',
    'NORWAY': 'EUROPE',
    'ZIMBABWE': 'AFRICA',
    'MOROCCO': 'AFRICA',
    'CHINA': 'ASIA',
    'GERMANY': 'EUROPE',
    'SOUTH AFRICA': 'AFRICA',
    'JAPAN': 'ASIA',
    'UAE': 'ASIA',
    'RUSSIA': 'EUROPE/ASIA',
    'KENYA': 'AFRICA',
    'GREECE': 'EUROPE',
    'POLAND': 'EUROPE',
    'AUSTRIA': 'EUROPE',
    'MALAYSIA': 'ASIA',
    'CZECH REPUBLIC': 'EUROPE',
    'KOREA': 'ASIA',
    'DRC': 'AFRICA',
    'CYPRUS': 'EUROPE',
    'KUWAIT': 'ASIA',
    'BRAZIL': 'SOUTH AMERICA',
    'CHILE': 'SOUTH AMERICA',
    'BULGARIA': 'EUROPE',
    'ISRAEL': 'ASIA',
    'NEW ZEALAND': 'OCEANIA',
    'UGANDA': 'AFRICA',
    'IRELAND': 'EUROPE',
    'SWEDEN': 'EUROPE',
    'QATAR': 'ASIA',
    'CONGO': 'AFRICA',
    'BURUNDI': 'AFRICA',
    'FINLAND': 'EUROPE',
    'GHANA': 'AFRICA',
    'OMAN': 'ASIA',
    'LEBANON': 'ASIA',
    'LUXEMBOURG': 'EUROPE',
    'IVORY COAST': 'AFRICA',
    'NAMIBIA': 'AFRICA',
    'ROMANIA': 'EUROPE',
    'THAILAND': 'ASIA',
    'SUDAN': 'AFRICA',
    'SINGAPORE': 'ASIA',
    'PORTUGAL': 'EUROPE',
    'MALAWI': 'AFRICA',
    'MALTA': 'EUROPE',
    'EGYPT': 'AFRICA',
    'PAKISTAN': 'ASIA',
    'TANZANIA': 'AFRICA',
    'ARGENTINA': 'SOUTH AMERICA',
    'SRI LANKA': 'ASIA',
    'BOTSWANA': 'AFRICA',
    'VIETNAM': 'ASIA',
    'BERMUDA': 'NORTH AMERICA',
    'TAIWAN': 'ASIA',
    'SENEGAL': 'AFRICA',
    'AFGHANISTAN': 'ASIA',
    'COLOMBIA': 'SOUTH AMERICA',
    'JAMAICA': 'NORTH AMERICA',
    'BAHRAIN': 'ASIA',
    'COSTA RICA': 'NORTH AMERICA',
    'COMOROS': 'AFRICA',
    'URUGUAY': 'SOUTH AMERICA',
    'SAUDI ARABIA': 'ASIA',
    'ETHIOPIA': 'AFRICA',
    'PHILIPPINES': 'ASIA',
    'ARMENIA': 'ASIA',
    'MEXICO': 'NORTH AMERICA',
    'MADAGASCAR': 'AFRICA',
    'SERBIA': 'EUROPE',
    'SLOVAKIA': 'EUROPE',
    'IRAN': 'ASIA',
    'SWAZILAND': 'AFRICA',
    'PERU': 'SOUTH AMERICA',
    'LITHUANIA': 'EUROPE',
    'ALGERIA': 'AFRICA',
    'SEYCHELLES': 'AFRICA',
    'MOZAMBIQUE': 'AFRICA',
    'UKRAINE': 'EUROPE',
    'NEPAL': 'ASIA',
    'CROATIA': 'EUROPE',
    'SOMALIA': 'AFRICA',
    'LIBERIA': 'AFRICA',
    'MAURITIUS': 'AFRICA',
    'GAMBIA': 'AFRICA',
    'ECUADOR': 'SOUTH AMERICA',
    'LATVIA': 'EUROPE',
    'MACEDONIA': 'EUROPE',
    'LESOTHO': 'AFRICA',
    'HUNGARY': 'EUROPE',
    'ICELAND': 'EUROPE',
    'VENEZUELA': 'SOUTH AMERICA',
    'ANGOLA': 'AFRICA',
    'SLOVENIA': 'EUROPE',
    'CAPE VERDE': 'AFRICA',
    'CAMEROON': 'AFRICA',
    'MONECASQUE': 'EUROPE',
    'TRINIDAD AND TOBAGO': 'NORTH AMERICA',
    'PAPUA NEW GUINEA': 'OCEANIA',
    'TURKEY': 'EUROPE/ASIA',
    'INDONESIA': 'ASIA',
    'ESTONIA': 'EUROPE',
    'BOSNIA': 'EUROPE',
    'YEMEN': 'ASIA',
    'NIGER': 'AFRICA',
    'ERITREA': 'AFRICA',
    'GEORGIA': 'ASIA',
    'BANGLADESH': 'ASIA',
    'BARBADOS': 'NORTH AMERICA',
    'TUNISIA': 'AFRICA',
    'DJIBOUTI': 'AFRICA',
    'DOMINICA': 'NORTH AMERICA',
    'CAMBODIA': 'ASIA',
    'MONTENEGRO': 'EUROPE',
    'JORDAN': 'ASIA',
    'KAZAKHSTAN': 'EUROPE/ASIA',
    'BENIN': 'AFRICA',
    'IRAQ': 'ASIA',
    'CHAD': 'AFRICA',
    'EL SALVADOR': 'NORTH AMERICA',
    'HAITI': 'NORTH AMERICA',
    'GUINEA': 'AFRICA',
    'BELARUS': 'EUROPE',
    'BOLIVIA': 'SOUTH AMERICA',
    'MYANMAR': 'ASIA'
}

In [37]:
for df in (train_processed, test_processed):
    df['continent'] = df['country'].map(continent_mapping)

## **4. Inspect the Engineered Dataset**

The preview and schema below show the training dataframe after feature creation. Rerun the notebook to populate these outputs. Apply the validation checks in Section 6 to both training and test data.

In [38]:
train_processed

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,first_trip_tz,cost_category,total nights,total_adults,mainland_nights_share,zanzibar_nights_share,adults_per_night,male_share,female_share,continent
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,High Cost,7,2.0,0.00,1.00,0.29,1.00,0.00,EUROPE
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,High Cost,7,2.0,0.00,1.00,0.29,0.50,0.50,EUROPE
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Higher Cost,12,2.0,0.50,0.50,0.17,0.50,0.50,NORTH AMERICA
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,Lower Cost,3,4.0,1.00,0.00,1.33,0.25,0.75,AFRICA
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Higher Cost,7,1.0,1.00,0.00,0.14,1.00,0.00,NORTH AMERICA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18473,tour_idmp5ciw50,KENYA,45-64,Alone,0.0,1.0,Other,Hunting Tourism,Others,Independent,...,No,Lower Cost,3,1.0,1.00,0.00,0.33,1.00,0.00,AFRICA
18474,tour_ideq0yncfs,KENYA,45-64,Alone,1.0,0.0,Meetings and Conference,Wildlife Tourism,Others,Independent,...,No,Lower Cost,2,1.0,1.00,0.00,0.50,0.00,1.00,AFRICA
18475,tour_idv7pz3vs8,UNITED STATES OF AMERICA,25-44,With Spouse and Children,2.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Independent,...,Yes,Higher Cost,9,3.0,1.00,0.00,0.33,0.33,0.67,NORTH AMERICA
18476,tour_idy6ydo00w,UNITED STATES OF AMERICA,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Package Tour,...,No,Higher Cost,17,2.0,0.76,0.24,0.12,0.50,0.50,NORTH AMERICA


In [39]:
train_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18478 entries, 0 to 18477
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18478 non-null  object 
 1   country                18478 non-null  object 
 2   age_group              18478 non-null  object 
 3   travel_with            18478 non-null  object 
 4   total_female           18478 non-null  float64
 5   total_male             18478 non-null  float64
 6   purpose                18478 non-null  object 
 7   main_activity          18478 non-null  object 
 8   info_source            18478 non-null  object 
 9   tour_arrangement       18478 non-null  object 
 10  package_transport_int  18478 non-null  object 
 11  package_accomodation   18478 non-null  object 
 12  package_food           18478 non-null  object 
 13  package_transport_tz   18478 non-null  object 
 14  package_sightseeing    18478 non-null  object 
 15  pa

In [40]:
test_processed

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,night_zanzibar,first_trip_tz,total nights,total_adults,mainland_nights_share,zanzibar_nights_share,adults_per_night,male_share,female_share,continent
0,tour_idynufedne,KOREA,25-44,Alone,0.0,1.0,Leisure and Holidays,Wildlife Tourism,Others,Independent,...,4,Yes,11,1.0,0.64,0.36,0.09,1.0,0.0,ASIA
1,tour_id9r3y5moe,UNITED KINGDOM,45-64,With Children,1.0,1.0,Leisure and Holidays,Conference Tourism,"Travel agent, tour operator",Package Tour,...,0,Yes,7,2.0,1.00,0.00,0.29,0.5,0.5,EUROPE
2,tour_idf6itml6g,ITALY,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,...,6,Yes,6,2.0,0.00,1.00,0.33,0.5,0.5,EUROPE
3,tour_id99u4znru,KENYA,25-44,Alone,0.0,1.0,Other,Beach Tourism,"Radio, TV, Web",Independent,...,4,No,7,1.0,0.43,0.57,0.14,1.0,0.0,AFRICA
4,tour_idj4i9urbx,ZAMBIA,25-44,Alone,0.0,1.0,Business,Wildlife Tourism,"Radio, TV, Web",Independent,...,0,No,6,1.0,1.00,0.00,0.17,1.0,0.0,AFRICA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6164,tour_id2deyfjhq,ZIMBABWE,25-44,Alone,0.0,1.0,Business,Wildlife Tourism,"Friends, relatives",Independent,...,0,No,2,1.0,1.00,0.00,0.50,1.0,0.0,AFRICA
6165,tour_idlenv2rio,DRC,25-44,Alone,0.0,1.0,Visiting Friends and Relatives,Hunting Tourism,"Friends, relatives",Independent,...,0,Yes,60,1.0,1.00,0.00,0.02,1.0,0.0,AFRICA
6166,tour_id7wwqrs0p,CANADA,25-44,Alone,0.0,1.0,Leisure and Holidays,Beach Tourism,"Friends, relatives",Package Tour,...,0,No,5,1.0,1.00,0.00,0.20,1.0,0.0,NORTH AMERICA
6167,tour_idx80vbw5a,CANADA,18-24,Alone,1.0,0.0,Visiting Friends and Relatives,Wildlife Tourism,"Friends, relatives",Independent,...,0,No,21,1.0,1.00,0.00,0.05,0.0,1.0,NORTH AMERICA


In [41]:
test_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6169 entries, 0 to 6168
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                6169 non-null   object 
 1   country                6169 non-null   object 
 2   age_group              6169 non-null   object 
 3   travel_with            6166 non-null   object 
 4   total_female           6167 non-null   float64
 5   total_male             6168 non-null   float64
 6   purpose                6169 non-null   object 
 7   main_activity          6169 non-null   object 
 8   info_source            6169 non-null   object 
 9   tour_arrangement       6169 non-null   object 
 10  package_transport_int  6169 non-null   object 
 11  package_accomodation   6169 non-null   object 
 12  package_food           6169 non-null   object 
 13  package_transport_tz   6169 non-null   object 
 14  package_sightseeing    6169 non-null   object 
 15  pack

## **5. Feature Engineering Summary**

| Feature | Definition | Interpretation |
| --- | --- | --- |
| `total nights` | Mainland nights + Zanzibar nights | Trip duration |
| `total_adults` | Female count + male count | Recorded group size |
| `mainland_nights_share` | Mainland nights / total nights | Mainland share of the stay |
| `zanzibar_nights_share` | Zanzibar nights / total nights | Zanzibar share of the stay |
| `adults_per_night` | Group size / total nights | Group-size and duration interaction |
| `male_share` | Male count / group size | Male share of the group |
| `female_share` | Female count / group size | Female share of the group |
| `continent` | Country lookup | Broader geographic origin |

The feature engineering process creates **8 candidate features**: 7 numerical columns and 1 categorical column. Starting from 21 training columns and 20 test columns, the expected schemas are **29 training columns** and **28 test columns**. Excluding the identifier and training target leaves 27 candidate predictors when all new features are retained.

These are expected counts from the transformations, not results from a new execution on the project data.

## **6. Validation Required Before Modeling**

Check the original counts for negative, missing, or non-finite values. With valid non-negative source counts and positive totals, each share should lie between 0 and 1, and each complementary pair should sum to 1 within floating-point tolerance.

Zero denominators intentionally produce missing values. Inspect their frequency and choose a modeling policy rather than replacing them with zero indiscriminately. XGBoost can handle missing numerical values; other estimators may require imputation fitted on training folds only.

Inspect `adults_per_night` for extreme values and all engineered numerical features for infinity. `info()` counts missing values but does not detect infinity. Check country-mapping coverage separately for training and test data.


## **7. Integration with the Modeling Pipeline**

Use `total_adults`, `male_share`, `total nights`, and `zanzibar_nights_share` as an initial candidate feature group. Retain both shares per pair in this notebook for exploration, but start modeling with only one per pair because the second is determined by the first whenever the total is valid. Test `adults_per_night` and `continent` as additional candidates.

Configure the model's numerical and categorical feature lists using the selected predictors. Include one share from each complementary pair in the initial model. Use `Tour_ID` as the record identifier and `cost_category` as the target.

The baseline notebook must be connected explicitly to these transformed dataframes or to a reusable feature builder. Apply identical transformations and column ordering at inference. Fit categorical encoders and any learned preprocessing within each training fold; save them with the feature definitions, target mapping, and model.

## **8. Experiment Conclusions**

The implementation represents group composition and destination allocation using shares of their totals. All-male versus all-female groups and mainland-only versus Zanzibar-only trips receive distinct, interpretable values.

Totals preserve size information, while shares describe composition. Zero-total cases are explicitly left undefined as `NaN`. The same feature definitions are applied to training and test datasets.

Set up, run the feature engineering file `src/features/builder.py` against the project's source datasets to inspect the calculated features. Model performance and feature importance require a subsequent training and evaluation experiment.   

The feature encoding file `src/features/encoding.py` will be created to isolate the categorical variable transformation logic and apply it securely and independently to prevent data leakage.

Modify `src/preprocessing.py` to adapt and align with the new feature columns introduced by the feature engineering pipeline.

## **9. Next Experiment: Generative Synthetic Data & Evaluation**

* Conduct the next experimental phase by generating synthetic data using advanced probabilistic and generative models (such as Copula-based methods or CTGAN) designed to faithfully preserve underlying statistical relationships, distributions, and feature correlations. This will be applied exclusively to the training folds to handle class imbalance while maintaining data integrity.

* Train and evaluate the model using the newly engineered candidate features on the balanced training folds, utilizing stratified splits, random seeds, model settings, and sample-weight policy. Fit encoders within each fold.

* Measure accuracy and macro F1 across folds, inspect class-level results, and evaluate feature groups separately. In particular, test whether shares add value beyond the original counts and totals, and whether `adults_per_night` adds value beyond trip duration and group size.

* Keep holdout and cross-validation metrics in separate variables. Save the finalized transformation pipeline, feature sets, and model artifacts based on consistent validation results.